In [ ]:
# Reload color scheme + style function if missing (kernel was restarted)
NAVY, WHITE, LGRAY, GRAY = "#0B1F3A", "#F8FAFC", "#CBD5E1", "#64748B"
TEAL, BLUE, PURP, AMBER, RED, GREEN = "#0D9488", "#0369A1", "#7C3AED", "#D97706", "#DC2626", "#16A34A"

def style_ax(ax, title):
    ax.set_facecolor("#0D2040")
    ax.tick_params(colors=LGRAY, labelsize=9)
    ax.xaxis.label.set_color(LGRAY); ax.yaxis.label.set_color(LGRAY)
    ax.title.set_color(WHITE)
    ax.set_title(title, fontweight="bold", fontsize=11, pad=10)
    for s in ax.spines.values(): s.set_edgecolor("#1E3A5F")
    ax.grid(True, alpha=0.15, color=LGRAY)

NameError: name 'BLUE' is not defined

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(NAVY)
style_ax(ax, "Material Detectability — Medipix3 vs EIGER2 @ 60 keV")

x = np.arange(len(MATERIALS))
width = 0.35

for i, chip in enumerate(TWO_CHIPS):
    sub = cat1_2chip[cat1_2chip["chip"]==chip].set_index("material").reindex(MATERIALS)
    offset = (i - 0.5) * width
    bars = ax.bar(x + offset, sub["cnr"], width, color=TWO_COLORS[chip],
                  alpha=0.85, label=CHIP_LABELS[chip])
    for bar, v in zip(bars, sub["cnr"]):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.1, f"{v:.1f}",
                ha="center", color=WHITE, fontsize=9, fontweight="bold")

ax.axhline(y=3, color=AMBER, linestyle="--", linewidth=1.5, label="Detectability threshold")
ax.set_xticks(x)
ax.set_xticklabels([f"{m.title()}\n({MATERIAL_THICKNESS[m]})" for m in MATERIALS])
ax.set_ylabel("CNR", color=WHITE)
ax.legend(facecolor="#0D2040", edgecolor="#1E3A5F", labelcolor=WHITE)

plt.tight_layout()
out = os.path.join(OUT_DIR, "mp3_vs_eiger2_cat1_materials.png")
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {out}")

# Win/loss tally
print("\nHead-to-head by material:")
for mat in MATERIALS:
    mp3 = cat1_2chip[(cat1_2chip["chip"]=="medipix3") & (cat1_2chip["material"]==mat)]["cnr"].values
    eig = cat1_2chip[(cat1_2chip["chip"]=="eiger2") & (cat1_2chip["material"]==mat)]["cnr"].values
    if len(mp3) and len(eig):
        winner = "EIGER2" if eig[0] > mp3[0] else "Medipix3"
        margin = abs(eig[0] - mp3[0]) / min(eig[0], mp3[0]) * 100
        print(f"  {mat:<10}: {winner} wins by {margin:.0f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.patch.set_facecolor(NAVY)
for ax in axes: style_ax(ax, "")

ax = axes[0]
ax.set_title("Detection Efficiency vs Energy", color=WHITE, fontweight="bold")
for chip in TWO_CHIPS:
    effs, es = [], []
    for E in ENERGIES:
        if (chip, E) in cat2_2chip:
            effs.append((cat2_2chip[(chip,E)]["hits"] / N_BASELINE) * 100)
            es.append(E)
    ax.plot(es, effs, marker="o", linewidth=2.5, markersize=10,
            color=TWO_COLORS[chip], label=CHIP_LABELS[chip])
    for e, v in zip(es, effs):
        ax.annotate(f"{v:.0f}%", xy=(e,v), xytext=(0,8), textcoords="offset points",
                    ha="center", color=WHITE, fontsize=8)
ax.set_xlabel("Energy (keV)", color=WHITE)
ax.set_ylabel("Efficiency (%)", color=WHITE)
ax.legend(facecolor="#0D2040", edgecolor="#1E3A5F", labelcolor=WHITE)

ax = axes[1]
ax.set_title("Energy Resolution vs Energy", color=WHITE, fontweight="bold")
for chip in TWO_CHIPS:
    sub = fwhm_2chip[fwhm_2chip["chip"]==chip].sort_values("energy")
    ax.plot(sub["energy"], sub["fwhm_keV"], marker="o", linewidth=2.5, markersize=10,
            color=TWO_COLORS[chip], label=CHIP_LABELS[chip])
    for _, row in sub.iterrows():
        ax.annotate(f"{row['fwhm_keV']:.2f}", xy=(row["energy"], row["fwhm_keV"]),
                    xytext=(0,8), textcoords="offset points",
                    ha="center", color=WHITE, fontsize=8)
ax.set_xlabel("Energy (keV)", color=WHITE)
ax.set_ylabel("FWHM (keV)", color=WHITE)
ax.legend(facecolor="#0D2040", edgecolor="#1E3A5F", labelcolor=WHITE)

plt.tight_layout()
out = os.path.join(OUT_DIR, "mp3_vs_eiger2_cat2_efficiency_resolution.png")
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {out}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(NAVY)
style_ax(ax, "Baggage Occlusion — Medipix3 vs EIGER2")

x_pos = range(len(OCCL_LEVELS))
for chip in TWO_CHIPS:
    sub = cat3_2chip[cat3_2chip["chip"]==chip].set_index("occlusion").reindex(OCCL_LEVELS)
    ax.plot(x_pos, sub["cnr"], marker="o", linewidth=2.5, markersize=10,
            color=TWO_COLORS[chip], label=CHIP_LABELS[chip])

ax.axhline(y=3, color=AMBER, linestyle="--", linewidth=1.5, label="Detectability threshold")
ax.set_xticks(x_pos)
ax.set_xticklabels(["None", "Sparse", "Medium", "Cluttered"])
ax.set_xlabel("Occlusion Level", color=WHITE)
ax.set_ylabel("CNR", color=WHITE)
ax.legend(facecolor="#0D2040", edgecolor="#1E3A5F", labelcolor=WHITE)

plt.tight_layout()
out = os.path.join(OUT_DIR, "mp3_vs_eiger2_cat3_occlusion.png")
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {out}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(NAVY)
style_ax(ax, "Degradation Stress Test — Medipix3 vs EIGER2")

x_pos = range(len(DEGR_STAGES))
for chip in TWO_CHIPS:
    fresh = degr_data.get((chip, "fresh"))
    if fresh is None: continue
    fresh_n = fresh["hits"]
    pcts = [(degr_data[(chip,s)]["hits"]/fresh_n*100) if degr_data.get((chip,s)) else None
            for s in DEGR_STAGES]
    ax.plot(x_pos, pcts, marker="o", linewidth=2.5, markersize=10,
            color=TWO_COLORS[chip], label=CHIP_LABELS[chip])

ax.axhline(y=80, color=AMBER, linestyle="--", linewidth=1.5, label="80% threshold")
ax.set_xticks(x_pos)
ax.set_xticklabels([s.title() for s in DEGR_STAGES])
ax.set_ylabel("Efficiency (% of fresh)", color=WHITE)
ax.legend(facecolor="#0D2040", edgecolor="#1E3A5F", labelcolor=WHITE)

plt.tight_layout()
out = os.path.join(OUT_DIR, "mp3_vs_eiger2_cat4_degradation.png")
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {out}")

In [ ]:
import math

TAU_EIGER2 = 100e-9   # confirmed dead time, <100ns
TAU_MP3 = 120e-9      # time-to-peak proxy (less certain, but same architecture type)

rates = np.logspace(3, 9, 200)
loss_mp3 = 1 - (rates/(1+rates*TAU_MP3)) / rates
loss_eiger2 = 1 - (rates/(1+rates*TAU_EIGER2)) / rates

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(NAVY)
style_ax(ax, "Count-Rate Dead-Time Loss — Medipix3 vs EIGER2 (both true counter architectures)")
ax.plot(rates, loss_mp3*100, color=BLUE, linewidth=2.5, label="Medipix3 (dead time ~120ns, proxy)")
ax.plot(rates, loss_eiger2*100, color=PURP, linewidth=2.5, label="EIGER2 (dead time <100ns, confirmed)")
ax.set_xscale("log")
ax.set_xlabel("True Photon Rate (counts/s/pixel)", color=WHITE)
ax.set_ylabel("Count Loss (%)", color=WHITE)
ax.legend(facecolor="#0D2040", edgecolor="#1E3A5F", labelcolor=WHITE)

plt.tight_layout()
out = os.path.join(OUT_DIR, "mp3_vs_eiger2_countrate.png")
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {out}")

In [ ]:
print("="*65)
print("MEDIPIX3 vs EIGER2 — Clean Head-to-Head (matched architectures)")
print("="*65)
print("\nCATEGORY 1 — Materials: EIGER2 wins every material (~10-15% margin)")
print("CATEGORY 2 — Efficiency: EIGER2 wins decisively (e.g. 75% vs 48% @ 60keV)")
print("CATEGORY 2 — Resolution: Medipix3 wins decisively (1.28 vs 2.12 keV @ 60keV)")
print("CATEGORY 3 — Occlusion: EIGER2 slightly ahead, both converge near threshold when cluttered")
print("CATEGORY 4 — Durability: Medipix3 wins by a very large margin (87% vs 8% at Heavy stress)")
print("CATEGORY 4 — Count rate: EIGER2's confirmed <100ns dead time outperforms Medipix3's ~120ns proxy")
print()
print("VERDICT: EIGER2 = better raw counting/throughput chip.")
print("         Medipix3 = better energy precision AND dramatically better durability.")
print("         Sensor thickness (750um vs 300um) is the root cause of both effects.")
print("="*65)